# Glints Job Scraper

Notebook ini melakukan scraping lowongan kerja dari **glints.com/id** menggunakan Selenium.

| # | Bagian | Deskripsi | Output |
|---|--------|-----------|--------|
| 1 | Import Library | Import semua dependensi Python | — |
| 2 | Konfigurasi | Set keyword, jumlah halaman, nama file output | — |
| 3 | Fungsi Utilitas | Fungsi pembantu yang dipakai di Stage 1 & 2 | — |
| 4 | Stage 1 | Buka list page, kumpulkan URL & lokasi tiap card | `unique_data` |
| 5 | Stage 2 | Buka tiap URL, scrape semua field dari halaman detail | `results` |
| 6 | Simpan Hasil | Gabungkan dan simpan ke CSV & Excel | `.csv` / `.xlsx` |

---
### Cara pakai

1. Sesuaikan `KEYWORD` dan `MAX_PAGES` di Bagian 2
2. Jalankan semua cell berurutan dari atas ke bawah
3. Hasil scraping tersimpan otomatis di folder yang sama dengan notebook

**Install dependencies (jalankan sekali):**
```
pip install selenium webdriver-manager pandas openpyxl tqdm
```

---
# BAGIAN 1 — Import Library

Semua library yang dibutuhkan diimport di sini.

| Library | Kegunaan |
|---------|----------|
| `pandas` | Membuat dan menyimpan DataFrame hasil scraping |
| `time` | Jeda antar request agar tidak diblokir sebagai bot |
| `re` | Regex untuk parsing URL dan teks |
| `tqdm` | Progress bar saat loop Stage 2 |
| `datetime` | Tidak dipakai langsung — disertakan untuk kebutuhan ekstensi |
| `selenium` | Otomasi browser Chrome untuk membuka dan membaca halaman web |
| `webdriver_manager` | Download ChromeDriver yang sesuai versi Chrome secara otomatis |

In [3]:
import pandas as pd
import time
import re
from tqdm import tqdm
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

print('Import selesai')

Import selesai


---
# BAGIAN 2 — Konfigurasi

Ubah parameter di sini sebelum menjalankan scraping.

| Parameter | Default | Keterangan |
|-----------|---------|------------|
| `KEYWORD` | `"Frontend Developer"` | Kata kunci pencarian lowongan |
| `MAX_PAGES` | `2` | Jumlah halaman list yang di-scrape |
| `DELAY` | `2.0` | Jeda (detik) antar aksi browser, naikkan jika sering timeout |
| `OUTPUT_CSV` | auto | Nama file output CSV, otomatis mengikuti `KEYWORD` |
| `OUTPUT_XLSX` | auto | Nama file output Excel, otomatis mengikuti `KEYWORD` |

`SEARCH_URL` dibangun otomatis dari `KEYWORD` dan tidak perlu diubah secara manual.

In [4]:
KEYWORD     = "Frontend Developer"   
MAX_PAGES   = 2                     
DELAY       = 2.0                   
OUTPUT_CSV  = "..\hasil_scraping\glints_jobs_Frontend Developer.csv"     
OUTPUT_XLSX = "..\hasil_scraping\glints_jobs_Frontend Developer.xlsx"    

SEARCH_URL = (
    "https://glints.com/id/opportunities/jobs/explore"
    f"?keyword={KEYWORD.replace(' ', '%20')}"
    "&country=ID&locationName=All+Cities%2FProvinces&lowestLocationLevel=1"
)

print(f"Keyword : {KEYWORD}")
print(f"Halaman : {MAX_PAGES}")
print(f"URL     : {SEARCH_URL}")

Keyword : Frontend Developer
Halaman : 2
URL     : https://glints.com/id/opportunities/jobs/explore?keyword=Frontend%20Developer&country=ID&locationName=All+Cities%2FProvinces&lowestLocationLevel=1


---
# BAGIAN 3 — Fungsi Utilitas

Fungsi-fungsi pembantu yang dipakai berulang di Stage 1 dan Stage 2.

| Fungsi | Deskripsi |
|--------|-----------|
| `init_driver()` | Buat instance Chrome WebDriver baru dengan konfigurasi anti-deteksi bot |
| `safe_text()` | Ambil teks dari elemen via CSS selector; kembalikan `default` bila elemen tidak ditemukan |
| `safe_find_all()` | Ambil semua elemen via CSS selector; kembalikan list kosong bila tidak ada |
| `scroll_down()` | Scroll halaman perlahan ke bawah agar konten lazy-load sempat dimuat |

### Catatan `init_driver()`

Beberapa opsi Chrome diaktifkan untuk menyamarkan bahwa browser dikendalikan skrip:
- `--disable-blink-features=AutomationControlled`, hilangkan flag `navigator.webdriver`
- `excludeSwitches: ['enable-automation']`, sembunyikan banner otomasi di Chrome
- Custom `user-agent`, agar header request terlihat seperti browser biasa
- CDP `Page.addScriptToEvaluateOnNewDocument`, override `navigator.webdriver` ke `undefined` di setiap halaman baru

In [6]:
def init_driver():
    """Buat Chrome WebDriver baru dengan pengaturan anti-deteksi bot"""
    options = Options()
    options.add_argument('--start-maximized')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option('excludeSwitches', ['enable-automation'])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.execute_cdp_cmd(
        'Page.addScriptToEvaluateOnNewDocument',
        {'source': "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
    )
    return driver


def safe_text(el, selector, default=''):
    """Ambil teks elemen berdasarkan CSS selector. Kembalikan default jika tidak ada"""
    try:
        return el.find_element(By.CSS_SELECTOR, selector).text.strip() or default
    except NoSuchElementException:
        return default


def safe_find_all(el, selector):
    """Ambil semua elemen berdasarkan CSS selector. Kembalikan list kosong jika tidak ada"""
    try:
        return el.find_elements(By.CSS_SELECTOR, selector)
    except:
        return []


def scroll_down(driver, steps=8, px=700, delay=0.5):
    """Scroll halaman perlahan ke bawah agar lazy-load konten muncul"""
    for _ in range(steps):
        driver.execute_script(f'window.scrollBy(0, {px});')
        time.sleep(delay)
    time.sleep(1.0)


print('Fungsi utilitas siap')

Fungsi utilitas siap


---
# BAGIAN 4 — Stage 1: Kumpulkan URL & Lokasi dari List Page

Stage 1 membuka halaman pencarian Glints dan mengumpulkan **URL** serta **lokasi**
dari setiap card lowongan yang tampil.

### Alur per halaman

1. Tunggu card muncul via `WebDriverWait` (selector utama >> fallback bila gagal)
2. Scroll halaman perlahan agar card lazy-load sempat termuat
3. Untuk setiap card:
   - Ambil `href` dari tag `<a>` yang mengarah ke `/id/opportunities/jobs/`
   - Ambil teks lokasi dari `[class*='LocationWrapper'] span[title]`
4. Navigasi ke halaman berikutnya via tombol *Next page* (tiga selector dicoba); bila semua gagal, modifikasi parameter `start=` di URL secara manual

### Strategi fallback selector

Glints kadang merilis perubahan class CSS. Dua selector disiapkan:
- **Utama:** `[class*='JobcardContainer'][class*='CompactJobCardWrapper']`
- **Fallback:** `[class*='JobcardContainer'], [class*='CompactJobCardWrapper']`

### Output

`unique_data` — list of dict `{'tautan': str, 'lokasi': str}` yang sudah dideduplikasi berdasarkan URL.

In [7]:
CARD_SEL      = "[class*='JobcardContainer'][class*='CompactJobCardWrapper']"
CARD_FALLBACK = "[class*='JobcardContainer'], [class*='CompactJobCardWrapper']"

driver = init_driver()
stage1_data = []  # list of dict {'tautan': ..., 'lokasi': ...}

try:
    driver.get(SEARCH_URL)
    time.sleep(DELAY)

    for page in range(1, MAX_PAGES + 1):
        print(f'[PAGE {page}/{MAX_PAGES}]')

        try:
            WebDriverWait(driver, 25).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, CARD_SEL))
            )
        except TimeoutException:
            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, CARD_FALLBACK))
                )
            except TimeoutException:
                print('  Card tidak ditemukan, berhenti.')
                break

        scroll_down(driver)

        cards = driver.find_elements(By.CSS_SELECTOR, CARD_SEL)
        if not cards:
            cards = driver.find_elements(By.CSS_SELECTOR, CARD_FALLBACK)

        for card in cards:
            # Ambil URL
            try:
                href = card.find_element(
                    By.CSS_SELECTOR, "a[href*='/id/opportunities/jobs/']"
                ).get_attribute('href') or ''
            except NoSuchElementException:
                try:
                    href = card.find_element(By.TAG_NAME, 'a').get_attribute('href') or ''
                except:
                    href = ''
            if href.startswith('/'):
                href = 'https://glints.com' + href

            # Ambil Lokasi dari card
            # Selector: div[class*='LocationWrapper'] > span[title]
            # Tiap span punya attribute title berisi nama kota/provinsi
            try:
                loc_spans = card.find_elements(
                    By.CSS_SELECTOR, "[class*='LocationWrapper'] span[title]"
                )
                lokasi = ', '.join(
                    [s.get_attribute('title') for s in loc_spans
                     if s.get_attribute('title')]
                )
            except:
                lokasi = ''

            if href:
                stage1_data.append({'tautan': href, 'lokasi': lokasi})

        print(f'  {len(cards)} card ditemukan')

        # Navigasi ke halaman berikutnya
        if page < MAX_PAGES:
            navigated = False
            for btn_sel in [
                "button[aria-label='Next page']",
                "li[title='Next Page'] button",
                "[class*='PaginationNext'] button"
            ]:
                try:
                    btn = driver.find_element(By.CSS_SELECTOR, btn_sel)
                    if btn.is_enabled():
                        driver.execute_script('arguments[0].click();', btn)
                        time.sleep(DELAY + 1)
                        navigated = True
                        break
                except NoSuchElementException:
                    continue
            if not navigated:
                cur = driver.current_url
                offset = page * 30
                new_url = re.sub(r'start=\d+', f'start={offset}', cur) \
                          if 'start=' in cur \
                          else cur + ('&' if '?' in cur else '?') + f'start={offset}'
                driver.get(new_url)
                time.sleep(DELAY + 1)

finally:
    driver.quit()

# Deduplikasi berdasarkan tautan
seen = set()
unique_data = []
for d in stage1_data:
    if d['tautan'] not in seen:
        seen.add(d['tautan'])
        unique_data.append(d)

print(f'\nTotal URL unik: {len(unique_data)}')

[PAGE 1/2]
  34 card ditemukan
[PAGE 2/2]
  34 card ditemukan

Total URL unik: 68


---
# BAGIAN 5 — Stage 2: Scrape Detail per URL

Stage 2 membuka setiap URL dari `unique_data` dan mengekstrak semua field detail lowongan.

## 5.1 Fungsi `scrape_job_detail()`

Fungsi ini menerima satu URL, membuka halaman detail, lalu mengekstrak field berikut:

| Field | Selector / Strategi Ekstraksi |
|-------|-------------------------------|
| `posisi` | `h1[aria-label="Job Title"]` >> fallback `[class*='JobOverViewTitle']` |
| `perusahaan` | `[class*='JobOverViewCompanyName']` |
| `gaji` | `innerText` dari `[class*='BasicSalary']` via JavaScript (agar text node angka ikut terbaca) |
| `jenis_pekerjaan` | Baris `JobOverViewInfo` yang mengandung tanda `·` atau keyword: *waktu, hybrid, remote, freelance* |
| `kategori` | Baris `JobOverViewInfo` yang punya link `/id/job-category/`, digabung dengan `>` |
| `pendidikan` | Baris `JobOverViewInfo` yang mengandung keyword: *sarjana, s1, s2, diploma, bachelor* |
| `pengalaman` | Baris `JobOverViewInfo`: jika kosong, dicari dari tag syarat requirement |
| `skill` | `[class*='Skillssc__TagContainer'] [class*='TagContentWrapper']` (tiga selector fallback) |
| `gender` | Tag requirement yang mengandung keyword: *pria, wanita, male, female* |
| `usia` | Tag requirement dengan angka + keyword *tahun/usia/age*, dikecualikan dari pengalaman & gender |
| `kualifikasi` | `[class*='ContentContainer']` |

Setiap driver dibuat baru per URL (`init_driver()`) dan ditutup di blok `finally`
agar tidak ada proses Chrome yang menggantung bila terjadi error.

In [8]:
def scrape_job_detail(job_url):
    """Buka halaman detail lowongan dan ekstrak semua field"""
    result = {
        'posisi'         : '',
        'perusahaan'     : '',
        'gaji'           : '',
        'jenis_pekerjaan': '',
        'kategori'       : '',
        'pendidikan'     : '',
        'pengalaman'     : '',
        'gender'         : '',
        'usia'           : '',
        'skill'          : '',
        'kualifikasi'    : '',
    }

    driver = None
    try:
        driver = init_driver()
        driver.get(job_url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.ID, '__next'))
        )
        time.sleep(2)
        scroll_down(driver, steps=5, px=600, delay=0.4)

        # Posisi
        result['posisi'] = safe_text(driver, 'h1[aria-label="Job Title"]')
        if not result['posisi']:
            result['posisi'] = safe_text(driver, "[class*='JobOverViewTitle']")

        # Perusahaan
        result['perusahaan'] = safe_text(driver, "[class*='JobOverViewCompanyName']")

        # Gaji : pakai innerText agar text node (angka) ikut terbaca
        try:
            sal_el = driver.find_element(By.CSS_SELECTOR, "[class*='BasicSalary']")
            raw = driver.execute_script('return arguments[0].innerText;', sal_el)
            result['gaji'] = ' '.join(raw.split()) if raw else ''
        except NoSuchElementException:
            result['gaji'] = ''

        # Info rows : jenis pekerjaan, kategori, pendidikan, pengalaman
        info_rows = safe_find_all(driver, "[class*='JobOverViewInfo-sc-b8dbys-9']")
        if not info_rows:
            info_rows = safe_find_all(driver, "[class*='JobOverViewInfo']")

        for row in info_rows:
            row_text = row.text.strip()
            if not row_text:
                continue

            # Kategori : baris yang punya link /id/job-category/
            cat_links = row.find_elements(By.CSS_SELECTOR, "a[href*='/id/job-category/']")
            if cat_links:
                result['kategori'] = ' > '.join(
                    [a.text.strip() for a in cat_links if a.text.strip()]
                )
                continue

            # Jenis Pekerjaan : baris bertanda · (misal: Penuh Waktu · Hybrid)
            jenis_kw = ['waktu', 'hybrid', 'remote', 'freelance', 'kontrak', 'penuh', 'paruh']
            if ('·' in row_text or any(k in row_text.lower() for k in jenis_kw)) \
                    and not result['jenis_pekerjaan']:
                result['jenis_pekerjaan'] = row_text
                continue

            # Pendidikan
            edu_kw = ['sarjana', 's1', 's2', 's3', 'diploma', 'd3', 'sma', 'bachelor', 'master']
            if any(k in row_text.lower() for k in edu_kw):
                result['pendidikan'] = row_text
                continue

            # Pengalaman
            exp_kw = ['tahun pengalaman', 'year experience', 'fresh graduate', 'pengalaman']
            if any(k in row_text.lower() for k in exp_kw):
                result['pengalaman'] = row_text
                continue

        # Skill
        skill_els = safe_find_all(driver,
            "[class*='Skillssc__TagContainer'] [class*='TagContentWrapper']")
        if not skill_els:
            skill_els = safe_find_all(driver,
                "[class*='Skillssc'] [class*='TagOverride'] [class*='TagContentWrapper']")
        if not skill_els:
            skill_els = safe_find_all(driver,
                "[class*='TagOverride'] [class*='TagContentWrapper']")
        result['skill'] = ', '.join([s.text.strip() for s in skill_els if s.text.strip()])

        # Gender & Usia : dari tag syarat requirement
        req_els = safe_find_all(driver,
            "[class*='JobRequirementssc__Tag'] [class*='TagContentWrapper']")
        req_texts = [r.text.strip() for r in req_els if r.text.strip()]
        
        gender_kw = ['pria', 'wanita', 'laki-laki', 'perempuan', 'male', 'female']
        result['gender'] = ', '.join(
            [t for t in req_texts if any(k in t.lower() for k in gender_kw)]
        )
        
        # Pengalaman : dari req_els (ada kata 'pengalaman' / 'experience')
        pengalaman_kw = ['pengalaman', 'experience', 'fresh graduate']
        req_pengalaman = ', '.join(
            [t for t in req_texts if any(k in t.lower() for k in pengalaman_kw)]
        )

        # Update result pengalaman jika belum terisi dari info_rows, 
        if req_pengalaman and not result['pengalaman']:
            result['pengalaman'] = req_pengalaman
        # Usia : ada angka + 'tahun'/'age'/'years', tapi BUKAN pengalaman/gender
        usia_kw = ['tahun', 'usia', 'umur', 'age', 'years old']
        result['usia'] = ', '.join(
            [t for t in req_texts
             if any(k in t.lower() for k in usia_kw)
             and re.search(r'\d', t)
             and not any(k in t.lower() for k in pengalaman_kw)   # bukan pengalaman
             and not any(k in t.lower() for k in gender_kw)]       # bukan gender
        )

        # Kualifikasi
        result['kualifikasi'] = safe_text(driver, "[class*='ContentContainer']")

    except Exception as e:
        print(f'  [ERR] {e}')
    finally:
        if driver:
            driver.quit()

    return result


print('Fungsi scrape_job_detail siap.')

Fungsi scrape_job_detail siap.


## 5.2 Jalankan Loop Stage 2

Iterasi semua URL di `unique_data`, panggil `scrape_job_detail()`, gabungkan hasilnya
dengan `lokasi` dari Stage 1, lalu simpan ke list `results`.

Progress ditampilkan per URL: posisi, perusahaan, lokasi, gaji, jenis, kategori, pendidikan, pengalaman, dan 60 karakter pertama skill.

In [9]:
# Jalankan scraping detail untuk setiap URL
results = []

for i, item in enumerate(tqdm(unique_data, desc='Scraping detail')):
    print(f'\n({i+1}/{len(unique_data)}) {item["tautan"]}')

    detail = scrape_job_detail(item['tautan'])

    # Gabung lokasi dari Stage 1 + detail dari Stage 2
    row = {'tautan': item['tautan'], 'lokasi': item['lokasi'], **detail}
    results.append(row)

    print(f'  Posisi     : {detail["posisi"]}')
    print(f'  Perusahaan : {detail["perusahaan"]}')
    print(f'  Lokasi     : {item["lokasi"]}')
    print(f'  Gaji       : {detail["gaji"]}')
    print(f'  Jenis      : {detail["jenis_pekerjaan"]}')
    print(f'  Kategori   : {detail["kategori"]}')
    print(f'  Pendidikan : {detail["pendidikan"]}')
    print(f'  Pengalaman : {detail["pengalaman"]}')
    print(f'  Skill      : {detail["skill"][:60]}')

    time.sleep(DELAY)

print(f'\nSelesai. Total: {len(results)} lowongan.')

Scraping detail:   0%|          | 0/68 [00:00<?, ?it/s]


(1/68) https://glints.com/id/opportunities/jobs/web-frontend-developer/15e34975-aa97-4a26-8bf0-ff4996cb7e73?utm_referrer=explore&traceInfo=75d0ad45-4d72-449f-b44b-df2145055026
  Posisi     : Web Frontend Developer
  Perusahaan : PT Adi Perdana Nusantara
  Lokasi     : Jakarta Utara, DKI Jakarta
  Gaji       : Rp6.000.000 - 7.000.000/Bulan
  Jenis      : Kontrak · Kerja di lokasi
  Kategori   : Komputer & Perangkat Lunak > Frontend Developer
  Pendidikan : Minimal Sarjana (S1)
  Pengalaman : 1 - 3 tahun pengalaman
  Skill      : React.js, Front-End Architecture, TypeScript, Frontend Devel


Scraping detail:   1%|▏         | 1/68 [00:21<23:52, 21.37s/it]


(2/68) https://glints.com/id/opportunities/jobs/frontend-developer/6b675c9f-7b21-46e9-aff5-2e4dda636417?utm_referrer=explore&traceInfo=75d0ad45-4d72-449f-b44b-df2145055026
  Posisi     : Frontend Developer
  Perusahaan : PT Dutakom Wibawa Putra (DWP)
  Lokasi     : Surabaya, Jawa Timur
  Gaji       : Rp5.200.000 - 5.500.000/Bulan
  Jenis      : Penuh Waktu · Kerja di lokasi
  Kategori   : Komputer & Perangkat Lunak > Frontend Developer
  Pendidikan : Minimal Diploma (D1 - D4)
  Pengalaman : 1 - 3 tahun pengalaman
  Skill      : PHP, Laravel, Frontend Development, React.js


Scraping detail:   3%|▎         | 2/68 [00:40<21:56, 19.95s/it]

...


(67/68) https://glints.com/id/opportunities/jobs/frontend-developer/88b263b9-19a3-46d4-b620-88f6ce317d73?traceInfo=027de1e1-5146-4667-a4e2-b9fc2faf2abb
  Posisi     : Frontend Developer
  Perusahaan : PT. Barrans Global Mandiri
  Lokasi     : Jakarta Pusat, DKI Jakarta
  Gaji       : 
  Jenis      : Kontrak · Kerja di lokasi
  Kategori   : Komputer & Perangkat Lunak > Frontend Developer
  Pendidikan : Minimal Sarjana (S1)
  Pengalaman : 1 - 3 tahun pengalaman
  Skill      : Vue.js, HTML, JavaScript, Node.js, TypeScript, React.js, Red


Scraping detail:  99%|█████████▊| 67/68 [21:05<00:18, 18.44s/it]


(68/68) https://glints.com/id/opportunities/jobs/frontend-developer/da4687f4-dba6-4390-994e-098f4a2249f9?traceInfo=027de1e1-5146-4667-a4e2-b9fc2faf2abb
  Posisi     : Frontend Developer
  Perusahaan : PT Aktivitas Insani Madani (AKSIMA)
  Lokasi     : Kab. Ngawi, Jawa Timur
  Gaji       : Rp1.500.000 - 2.500.000/Bulan
  Jenis      : Penuh Waktu · Kerja di lokasi
  Kategori   : Komputer & Perangkat Lunak > Frontend Developer
  Pendidikan : Minimal Sarjana (S1)
  Pengalaman : 
  Skill      : JavaScript, React.js, CSS, Web Design, HTML, Flutter


Scraping detail: 100%|██████████| 68/68 [21:23<00:00, 18.87s/it]


Selesai. Total: 68 lowongan.


---
# BAGIAN 6 — Simpan Hasil

`results` dikonversi ke DataFrame dengan urutan kolom yang sudah ditentukan di `col_order`.
Kolom yang mungkin tidak ada di hasil scraping ditambahkan sebagai string kosong
agar struktur file output selalu konsisten.

File disimpan dalam dua format:
- **CSV** (`utf-8-sig`) : kompatibel dengan Excel di Windows tanpa masalah encoding
- **Excel** (`.xlsx`) : siap dibuka langsung di spreadsheet

In [10]:
col_order = [
    'tautan', 'posisi', 'perusahaan', 'lokasi', 'gaji',
    'jenis_pekerjaan', 'kategori', 'pendidikan', 'pengalaman',
    'gender', 'usia', 'skill', 'kualifikasi'
]

final_df = pd.DataFrame(results)
for col in col_order:
    if col not in final_df.columns:
        final_df[col] = ''
final_df = final_df[col_order]

final_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
final_df.to_excel(OUTPUT_XLSX, index=False, engine='openpyxl')

print(f'Tersimpan: {len(final_df)} lowongan')
print(f'CSV  → {OUTPUT_CSV}')
print(f'XLSX → {OUTPUT_XLSX}')

Tersimpan: 68 lowongan
CSV  → ..\hasil_scraping\glints_jobs_Frontend Developer.csv
XLSX → ..\hasil_scraping\glints_jobs_Frontend Developer.xlsx


---
# BAGIAN 7 — Preview & Inspeksi Hasil

Dua cell di bawah untuk memeriksa hasil scraping secara cepat sebelum file digunakan lebih lanjut.

## 7.1 Preview 5 Baris Pertama

In [11]:
final_df.head()

,tautan,posisi,perusahaan,lokasi,gaji,jenis_pekerjaan,kategori,pendidikan,pengalaman,gender,usia,skill,kualifikasi
0,https://glints.com/id/opportunities/jobs/web-f...,Web Frontend Developer,PT Adi Perdana Nusantara,"Jakarta Utara, DKI Jakarta",Rp6.000.000 - 7.000.000/Bulan,Kontrak · Kerja di lokasi,Komputer & Perangkat Lunak > Frontend Developer,Minimal Sarjana (S1),1 - 3 tahun pengalaman,,,"React.js, Front-End Architecture, TypeScript, ...",Web Frontend Developer\nQualifications:\nBache...
1,https://glints.com/id/opportunities/jobs/front...,Frontend Developer,PT Dutakom Wibawa Putra (DWP),"Surabaya, Jawa Timur",Rp5.200.000 - 5.500.000/Bulan,Penuh Waktu · Kerja di lokasi,Komputer & Perangkat Lunak > Frontend Developer,Minimal Diploma (D1 - D4),1 - 3 tahun pengalaman,,,"PHP, Laravel, Frontend Development, React.js",Tugas dan Tanggung Jawab\n\n1. Melanjutkan tim...
2,https://glints.com/id/opportunities/jobs/front...,Frontend Developer,PT. Barrans Global Mandiri,"Jakarta Pusat, DKI Jakarta",,Kontrak · Kerja di lokasi,Komputer & Perangkat Lunak > Frontend Developer,Minimal Sarjana (S1),1 - 3 tahun pengalaman,,,"Vue.js, HTML, JavaScript, Node.js, TypeScript,...",- Minimum bachelor’s degree in Computer Scienc...
3,https://glints.com/id/opportunities/jobs/front...,Frontend Developer,PT GDC Multi Sarana (Jakarta),"Jakarta Selatan, DKI Jakarta",Rp10.000.000 - 15.000.000/Bulan,Penuh Waktu · Kerja di lokasi,Komputer & Perangkat Lunak > Frontend Developer,Minimal SMA/SMK,3 - 5 tahun pengalaman,,22-38 tahun,"Frontend Development, Dart, Flutter, React Nat...",Kualifikasi:\nMemiliki pengalaman yang kuat me...
4,https://glints.com/id/opportunities/jobs/front...,Frontend Developer,Delta HQ,"Surabaya, Jawa Timur",,Penuh Waktu · Kerja di lokasi,Komputer & Perangkat Lunak > Frontend Developer,Minimal Diploma (D1 - D4),1 - 3 tahun pengalaman,,,"JavaScript, Frontend Development, React.js",Company Description\nWe are PropTech company b...


## 7.2 Info Kolom & Missing Values

Tampilkan tipe data tiap kolom dan jumlah nilai non-null : berguna untuk mengidentifikasi
field mana yang banyak kosong (misalnya `gaji` atau `usia` yang memang sering tidak diisi perusahaan).

In [12]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   tautan           68 non-null     object
 1   posisi           68 non-null     object
 2   perusahaan       68 non-null     object
 3   lokasi           68 non-null     object
 4   gaji             68 non-null     object
 5   jenis_pekerjaan  68 non-null     object
 6   kategori         68 non-null     object
 7   pendidikan       68 non-null     object
 8   pengalaman       68 non-null     object
 9   gender           68 non-null     object
 10  usia             68 non-null     object
 11  skill            68 non-null     object
 12  kualifikasi      68 non-null     object
dtypes: object(13)
memory usage: 7.0+ KB
